# Supplementary Table 12 — L2G against naive prioritisation

Built from the L2G gold-standard training set, `data/l2g_training_set/`. That directory was
recovered from `~/Projects/EGL_and_training_set/archive/gentropy_paper/data/` on 2026-08-19; until
then this table was recorded as blocked on a missing input. It is not yet fetched by the download
notebook — see `GAPS.md`.

**ST12** compares L2G against the naive prioritisation methods over the full gold-standard set
(132,970 labelled credible-set/gene pairs, 8,520 positive).

This notebook used to write a second sheet, `ST13_effector_gene_list.csv`, from the positive half
of the gold standard. **Retired 2026-09-02.** It was never the effector gene list — the positives
are that list after it has been joined onto credible sets, 1,704 pairs against the 42,288 the list
itself carries — and `tab:st13` was removed from the manuscript on 2026-08-24 in any case. The two
artefacts it stood in for are now sheets of their own: `05_effector_gene_list.ipynb` writes the
effector gene list from the EGL parquet, and `06_l2g_training_set.ipynb` writes the full labelled
training set, positives and negatives together. See the chapter README.

In [1]:
import numpy as np
import pandas as pd
import pyarrow.dataset as pads

from manuscript_methods import paper

TRAINING_SET = paper.ROOT / "data" / "l2g_training_set" / "20250625_gentropy_paper_v1"

# The thresholds 05_l2g_prioritised_genes.ipynb uses for the same evidence flags.
CLPP, H4, VEP_PAV = 0.01, 0.8, 0.66

gold = pads.dataset(str(TRAINING_SET), format="parquet").to_table().to_pandas()
gold["positive"] = (gold["goldStandardSet"] == "positive").astype(int)
print(f"gold-standard pairs: {len(gold):,}  positive: {int(gold['positive'].sum()):,}")

gold-standard pairs: 132,970  positive: 8,520


## Supplementary Table 12 — L2G against naive prioritisation

One row per prioritisation rule, scored against the gold standard as a 2x2 table. The L2G score
comes from the prediction set the pipeline uses; the four evidence flags come from the feature
matrix, thresholded exactly as the prioritised-gene table does.

In [2]:
scores = (
    pads.dataset(str(paper.ROOT / "data/25.06/irene_1208_l2g_predictions"), format="parquet")
    .to_table(columns=["studyLocusId", "geneId", "score"])
    .to_pandas()
)
features = (
    pads.dataset(str(paper.release("l2g_feature_matrix")), format="parquet")
    .to_table(
        columns=[
            "studyLocusId",
            "geneId",
            "eQtlColocClppMaximum",
            "eQtlColocH4Maximum",
            "pQtlColocClppMaximum",
            "pQtlColocH4Maximum",
            "vepMaximum",
            "distanceSentinelTssNeighbourhood",
        ]
    )
    .to_pandas()
)

labelled = gold.merge(scores, on=["studyLocusId", "geneId"], how="left").merge(
    features, on=["studyLocusId", "geneId"], how="left"
)
assert len(labelled) == len(gold), "the joins changed the number of labelled pairs"
score = labelled["score"].fillna(0)

evidence = {
    "L2G>=0.5": score >= 0.5,
    "L2G>=0.05": score >= 0.05,
    "L2G>=0.8": score >= 0.8,
    "eQTL_coloc": (labelled["eQtlColocClppMaximum"].fillna(0) >= CLPP)
    | (labelled["eQtlColocH4Maximum"].fillna(0) >= H4),
    "pQTL_coloc": (labelled["pQtlColocClppMaximum"].fillna(0) >= CLPP)
    | (labelled["pQtlColocH4Maximum"].fillna(0) >= H4),
    "PAV": labelled["vepMaximum"].fillna(0) >= VEP_PAV,
    "Nearest to TSS": labelled["distanceSentinelTssNeighbourhood"].fillna(0) == 1,
}
# "Combined" is the manuscript's own prioritisation rule: every gene scoring L2G >= 0.5, plus the
# top-scoring gene of any credible set that has none. That is exactly what
# 01-data-preparation/05_l2g_prioritised_genes.ipynb writes, so the table is read rather than
# reimplemented — recomputing the rule from raw scores gives 6,798 / 123,051 / 1,399 / 1,722
# instead of the published 6,796 / 123,069 / 1,381 / 1,724, because the derived table also applies
# the protein-coding restriction.
prioritised = pd.read_parquet(
    paper.derived("prioritised_genes_per_cs"), columns=["studyLocusId", "geneId"]
).drop_duplicates()
prioritised["prioritised"] = True
evidence["Combined"] = (
    labelled.merge(prioritised, on=["studyLocusId", "geneId"], how="left")["prioritised"]
    .fillna(False)
    .to_numpy(dtype=bool)
)

/var/folders/p5/4t9crp1563l792qz8xz_3x5h0000gq/T/ipykernel_22334/3718819807.py:52: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


In [3]:
def confusion(predicted, actual):
    """The 2x2 table and the rates Supplementary Table 12 reports."""
    predicted, actual = np.asarray(predicted), np.asarray(actual).astype(bool)
    tp = int((predicted & actual).sum())
    tn = int((~predicted & ~actual).sum())
    fp = int((predicted & ~actual).sum())
    fn = int((~predicted & actual).sum())
    sensitivity = tp / (tp + fn) if tp + fn else np.nan
    specificity = tn / (tn + fp) if tn + fp else np.nan
    ppv = tp / (tp + fp) if tp + fp else np.nan
    return {
        "Evidence": None,
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "Sensitivity (recall)": sensitivity,
        "Specificity (selectivity)": specificity,
        "PPV (precision)": ppv,
        "FDR": 1 - ppv if ppv == ppv else np.nan,
        "Balanced_accuracy": (sensitivity + specificity) / 2,
    }


st12 = pd.DataFrame([{**confusion(mask, labelled["positive"]), "Evidence": name} for name, mask in evidence.items()])
st12 = st12[
    [
        "Evidence",
        "TP",
        "TN",
        "FP",
        "FN",
        "Sensitivity (recall)",
        "Specificity (selectivity)",
        "PPV (precision)",
        "FDR",
        "Balanced_accuracy",
    ]
]
st12.to_csv(paper.ROOT / "chapters/06-supplementary-tables/sheets/ST12_l2g_performance.csv", index=False)
st12.round(6)

,Evidence,TP,TN,FP,FN,Sensitivity (recall),Specificity (selectivity),PPV (precision),FDR,Balanced_accuracy
0,L2G>=0.5,6267,123785,665,2253,0.735563,0.994656,0.904068,0.095932,0.865110
1,L2G>=0.05,7501,117999,6451,1019,0.880399,0.948164,0.537629,0.462371,0.914281
2,L2G>=0.8,4498,124288,162,4022,0.527934,0.998698,0.965236,0.034764,0.763316
3,eQTL_coloc,2966,119501,4949,5554,0.348122,0.960233,0.374732,0.625268,0.654178
4,pQTL_coloc,1198,124116,334,7322,0.140610,0.997316,0.781984,0.218016,0.568963
5,PAV,1761,124021,429,6759,0.206690,0.996553,0.804110,0.195890,0.601621
6,Nearest to TSS,5484,122190,2260,3036,0.643662,0.981840,0.708161,0.291839,0.812751
7,Combined,6796,123069,1381,1724,0.797653,0.988903,0.831112,0.168888,0.893278


In [4]:
# Against the published sheet.
published = pd.DataFrame(
    [
        ("L2G>=0.5", 6267, 123785, 665, 2253),
        ("L2G>=0.05", 7501, 117999, 6451, 1019),
        ("L2G>=0.8", 4498, 124288, 162, 4022),
        ("eQTL_coloc", 2966, 119501, 4949, 5554),
        ("pQTL_coloc", 1198, 124116, 334, 7322),
        ("PAV", 1761, 124021, 429, 6759),
        ("Nearest to TSS", 5484, 122190, 2260, 3036),
        ("Combined", 6796, 123069, 1381, 1724),
    ],
    columns=["Evidence", "TP", "TN", "FP", "FN"],
)
check = published.merge(st12[["Evidence", "TP", "TN", "FP", "FN"]], on="Evidence", suffixes=("_published", "_computed"))
check["matches"] = [
    all(row[f"{c}_published"] == row[f"{c}_computed"] for c in ["TP", "TN", "FP", "FN"]) for _, row in check.iterrows()
]
print(f"rows reproducing exactly: {int(check['matches'].sum())} of {len(check)}")
check

rows reproducing exactly: 8 of 8


,Evidence,TP_published,TN_published,FP_published,FN_published,TP_computed,TN_computed,FP_computed,FN_computed,matches
0,L2G>=0.5,6267,123785,665,2253,6267,123785,665,2253,True
1,L2G>=0.05,7501,117999,6451,1019,7501,117999,6451,1019,True
2,L2G>=0.8,4498,124288,162,4022,4498,124288,162,4022,True
3,eQTL_coloc,2966,119501,4949,5554,2966,119501,4949,5554,True
4,pQTL_coloc,1198,124116,334,7322,1198,124116,334,7322,True
5,PAV,1761,124021,429,6759,1761,124021,429,6759,True
6,Nearest to TSS,5484,122190,2260,3036,5484,122190,2260,3036,True
7,Combined,6796,123069,1381,1724,6796,123069,1381,1724,True


### The `Combined` row

`Combined` is the manuscript's prioritisation rule — L2G >= 0.5, plus the top-scoring gene of any
credible set with no gene above 0.5 — and the pipeline already materialises it as
`prioritised_genes_per_cs`. Reading that table reproduces the published row exactly. Reimplementing
the rule from the raw scores does not: it misses the protein-coding restriction and lands two
positives and eighteen negatives out.